In [1]:
from crewai_tools import SerperDevTool
import os
from dotenv import load_dotenv

load_dotenv()
search_tool = SerperDevTool()

In [2]:
from crewai.tools import tool

@tool("retrieve_return_policy")
def retrieve_return_policy(query: str) -> str:
    """
    Returns the company's return policy.

    Args:
        query (str): Customer's inquiry about the return policy as a python string
    """
    return (
        "Our return policy states that if the cost to fix any damage to the item is estimated to be LESS THAN $300, "
        "you have a receipt of purchase, and you bought it within the last 60 days, "
        "you can return it and receive store credit."
    )

@tool("retrieve_shipping_policy")
def retrieve_shipping_policy() -> str:
    """
    description: Returns the company's shipping policy for handling returns.
    """
    return (
        "Our shipping policy for returns is as follows: "
        "Processing and delivery of returned items typically take 5-7 business days. "
        "Customers are responsible for a flat return shipping fee of $5 unless the item is defective. "
        "Tracking numbers are provided once the return is processed and shipped."
    )

@tool("retrieve_complaint_protocol")
def retrieve_complaint_protocol() -> str:
    """
    description: Provides the company's complaint resolution process for return-related issues.
    """
    return (
        "Our complaint resolution process for return-related issues is as follows: "
        "Complaints are acknowledged within 24 hours of submission. "
        "The resolution steps include: "
        "1) Reviewing the complaint and validating the return case, "
        "2) Contacting the customer for clarification or additional details if needed, and "
        "3) Providing a resolution, such as a refund, replacement, or escalation. "
        "If unresolved within 3 business days, the case is escalated to a senior support representative for priority handling."
    )

In [3]:
import yaml

# Load agents from YAML file
with open('multi_agents.yaml', 'r') as agents_file:
    agents_config = yaml.safe_load(agents_file)

# Load tasks from YAML file
with open('multi_colab_task.yaml', 'r') as tasks_file:
    tasks_config = yaml.safe_load(tasks_file)

## Our Multi-Agent Crew

In [4]:
from crewai import Agent, Task, Crew, Process

# Create agents
customer_support_agent = Agent(
    role=agents_config['customer_support_agent']['role'],
    goal=agents_config['customer_support_agent']['goal'],
    backstory=agents_config['customer_support_agent']['backstory'],
    tools=[],
    verbose=True,
    allow_delegation=True
)

technical_support_engineer = Agent(
    role=agents_config['technical_support_engineer']['role'],
    goal=agents_config['technical_support_engineer']['goal'],
    backstory=agents_config['technical_support_engineer']['backstory'],
    tools=[search_tool],
    verbose=True,
    allow_delegation=False
)

returns_assistant = Agent(
    role=agents_config['returns_assistant']['role'],
    goal=agents_config['returns_assistant']['goal'],
    backstory=agents_config['returns_assistant']['backstory'],
    tools=[retrieve_return_policy, retrieve_shipping_policy, retrieve_complaint_protocol],
    verbose=True,
    allow_delegation=False
)

# Create tasks
support_task = Task(
    description=tasks_config['support_task']['description'],
    expected_output=tasks_config['support_task']['expected_output']
)

# Create and execute the crew
crew = Crew(
    agents=[
        technical_support_engineer,
        returns_assistant
    ],
    tasks=[support_task],  # Tasks will be planned and delegated by the manager agent
    process=Process.hierarchical,
    manager_agent=customer_support_agent,
    verbose=True,
    planning=True,
    memory=True
)

print(crew)

entity_type='crew' name='crew' cache=True tasks=[Task(description=Handle the following support request from a user: {customer_inquiry}
, expected_output=An email response to the users support request.)] agents=[Agent(role=Technical Support Engineer
, goal=Address technical problems related to the company's products or services, including troubleshooting malfunctions and resolving defects. \n\n NOTE: If you need to assess damage, you can use the internet search tool to do so.
, backstory=You are a highly skilled technical expert focused on identifying and resolving issues to enhance product reliability and customer satisfaction. You work closely with the Customer Support Agent and Product Manager for technical escalations.
), Agent(role=Returns Assistant
, goal=Evaluate whether an item meets the company's return policy by verifying the following:
  - Condition of the item (e.g., unused, undamaged, etc.)
  - Eligibility based on purchase date and return window
  - Compliance with specifi

In [6]:
results = await crew.kickoff_async({"customer_inquiry": "I bought a computer and it is damaged, the screen is slightly cracked. Can I return it?"})

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.2                                                                                        │
│  Latest version:  1.15.4                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9ce8cd9e-5a7a-4c5c-bf0d-1d7bc3aa3b45                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-07-18 22:24:34][INFO]: Planning the crew execution


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Handle the following support request from a user: I bought a computer and it   │
│  is damaged, the screen is slightly cracked. Can I return it?                                                   │
│                                                                                                                 │
│                  "task_description": Handle the following support request from a user: I bought a computer and  │
│  it is damaged, the screen is slightly cracked. Can I return it?                                                │
│                                                                                                                 │
│                  "task_expected_output": An email response to the users support request.                        │
│                  "agent": None                                                                                  │
│                  "agent_goal": None                                                                             │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  ID: 3b147a7e-6413-4a69-acb7-4037d7d166cd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Based on these tasks summary:                                                                            │
│                  Task Number 1 - Handle the following support request from a user: I bought a computer and it   │
│  is damaged, the screen is slightly cracked. Can I return it?                                                   │
│                                                                                                                 │
│                  "task_description": Handle the following support request from a user: I bought a computer and  │
│  it is damaged, the screen is slightly cracked. Can I return it?                                                │
│                                                                                                                 │
│                  "task_expected_output": An email response to the users support request.                        │
│                  "agent": None                                                                                  │
│                  "agent_goal": None                                                                             │
│                  "task_tools": []                                                                               │
│                  "agent_tools": "agent has no tools"                                                            │
│   Create the most descriptive plan based on the tasks descriptions, tools available, and agents' goals for      │
│  them to execute their goals with perfection.                                                                   │
│  Agent: Task Execution Planner                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Customer Support Agent                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Subject: Regarding Your Computer Return Request for Screen Crack Issue                                         │
│                                                                                                                 │
│  Dear Customer,                                                                                                 │
│                                                                                                                 │
│  Thank you for reaching out to us regarding your computer purchase. We understand the concern caused by the     │
│  slight crack on the screen, and we’re here to help.                                                            │
│                                                                                                                 │
│  Return eligibility for damaged items like this typically depends on our return and damage policies.            │
│  Generally, if a product is reported with damage promptly, it may be processed as a support or return case. To  │
│  assist you further, could you please provide your order details and photos of the cracked screen? This         │
│  information will help us assess the situation and guide you through the next steps.                            │
│                                                                                                                 │
│  We appreciate your patience and look forward to resolving this for you. Please reply at your convenience with  │
│  the requested information or any additional questions you may have.                                            │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│  [Your Name]                                                                                                    │
│  Customer Support Team                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Handle the following support request from a user: I bought a computer and it is damaged, the screen is   │
│  slightly cracked. Can I return it?                                                                             │
│  1. Read the user’s support request carefully and identify the core issue: the user bought a computer, it       │
│  arrived damaged, and they want to know whether it can be returned.                                             │
│  2. Determine the response objective: provide a clear email response that addresses return eligibility for a    │
│  damaged item, while remaining polite, professional, and helpful.                                               │
│  3. Since no tools are available, rely only on the information contained in the task description and avoid      │
│  inventing specific company policies, timelines, or procedures that are not provided.                           │
│  4. Draft the email in a customer-support tone that acknowledges the damage and the user’s concern with         │
│  empathy.                                                                                                       │
│  5. Answer the user’s question directly by stating that return eligibility depends on the seller’s return and   │
│  damage policy, and that damaged items are generally handled as a support/return case if reported promptly.     │
│  6. Include a request for any necessary follow-up information that would normally be needed to process the      │
│  case, such as order details and photos of the cracked screen, while keeping the message generic because no     │
│  tools or policy details are available.                                                                         │
│  7. Keep the response concise, clear, and action-oriented, so the user knows what to do next.                   │
│  8. End the email with a courteous closing and an invitation for the user to reply with additional details if   │
│  needed.                                                                                                        │
│  9. Review the email for tone, clarity, and completeness to ensure it fully satisfies the expected output: an   │
│  email response to the user’s support request.                                                                  │
│  Agent: Customer Support Agent                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9ce8cd9e-5a7a-4c5c-bf0d-1d7bc3aa3b45                                                                       │
│  Final Output: Subject: Regarding Your Computer Return Request for Screen Crack Issue                           │
│                                                                                                                 │
│  Dear Customer,                                                                                                 │
│                                                                                                                 │
│  Thank you for reaching out to us regarding your computer purchase. We understand the concern caused by the     │
│  slight crack on the screen, and we’re here to help.                                                            │
│                                                                                                                 │
│  Return eligibility for damaged items like this typically depends on our return and damage policies.            │
│  Generally, if a product is reported with damage promptly, it may be processed as a support or return case. To  │
│  assist you further, could you please provide your order details and photos of the cracked screen? This         │
│  information will help us assess the situation and guide you through the next steps.                            │
│                                                                                                                 │
│  We appreciate your patience and look forward to resolving this for you. Please reply at your convenience with  │
│  the requested information or any additional questions you may have.                                            │
│                                                                                                                 │
│  Best regards,                                                                                                  │
│  [Your Name]                                                                                                    │
│  Customer Support Team                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_90228/1857657993.py:1: RuntimeWarning: coroutine 'Crew.kickoff_async' was never awaited
  results = await crew.kickoff_async({"customer_inquiry": "I bought a computer and it is damaged, the screen is slightly cracked. Can I return it?"})
